In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

# ============================================================================
# VQ Layer - Vector Quantization
# ============================================================================
class VectorQuantizer(nn.Module):
    def __init__(self, num_embeddings, embedding_dim, commitment_cost=0.25):
        super().__init__()
        self.embedding_dim = embedding_dim
        self.num_embeddings = num_embeddings
        self.commitment_cost = commitment_cost

        self.embeddings = nn.Embedding(num_embeddings, embedding_dim)
        self.embeddings.weight.data.uniform_(-1/num_embeddings, 1/num_embeddings)

    def forward(self, z):
        input_shape = z.shape
        flat_z = z.reshape(-1, self.embedding_dim)

        distances = (flat_z.pow(2).sum(1, keepdim=True)
                    - 2 * flat_z @ self.embeddings.weight.t()
                    + self.embeddings.weight.pow(2).sum(1, keepdim=True).t())

        encoding_indices = distances.argmin(1).unsqueeze(1)
        quantized = self.embeddings(encoding_indices).view(input_shape)

        e_latent_loss = F.mse_loss(quantized.detach(), z)
        q_latent_loss = F.mse_loss(quantized, z.detach())
        loss = q_latent_loss + self.commitment_cost * e_latent_loss

        quantized = z + (quantized - z).detach()

        return quantized, loss, encoding_indices.view(input_shape[:-1])

# ============================================================================
# Enhanced HRM Block with Multiple Cycles and Feature Aggregation
# ============================================================================
class HRMBlock(nn.Module):
    """
    Enhanced HRM block that:
    - Processes patches through multiple HRM cycles
    - Aggregates features from all cycles (not just final state)
    - Uses learnable projections for better capacity
    """
    def __init__(self, embed_dim, h_cycle=3, l_cycle=2, aggregate_features=True):
        super().__init__()
        self.h_cycle = h_cycle
        self.l_cycle = l_cycle
        self.embed_dim = embed_dim
        self.aggregate_features = aggregate_features

        # HRM cells
        self.low = nn.GRUCell(embed_dim, embed_dim, bias=False)
        self.high = nn.GRUCell(embed_dim, embed_dim, bias=False)

        # Feature projection layers
        self.input_proj = nn.Linear(embed_dim, embed_dim)

        if aggregate_features:
            # Aggregate features from all cycles
            total_cycles = h_cycle * l_cycle
            self.feature_aggregator = nn.Sequential(
                nn.Linear(embed_dim * (total_cycles + 1), embed_dim * 2),
                nn.ReLU(),
                nn.Linear(embed_dim * 2, embed_dim)
            )

        self.output_proj = nn.Linear(embed_dim, embed_dim)

    def forward(self, patches):
        """
        Args:
            patches: [B, N, C] where N is number of patches
        Returns:
            features: [B, N, C] aggregated features
        """
        B, N, C = patches.shape
        device = patches.device

        # Project input
        patches = self.input_proj(patches)

        # Store features from each cycle
        if self.aggregate_features:
            all_features = []

        # Initialize states
        z_l = torch.zeros(B, C, device=device)
        z_h = torch.zeros(B, C, device=device)

        # Process each patch
        patch_features = []
        for idx in range(N):
            patch = patches[:, idx, :]  # [B, C]

            cycle_features = [patch]

            # Run HRM cycles
            for cycle in range(self.h_cycle * self.l_cycle):
                if cycle % self.h_cycle == 0:
                    z_h = self.high(patch, z_l)
                z_l = self.low(patch, z_h)

                if self.aggregate_features:
                    cycle_features.append(z_l)

            # Aggregate features from all cycles for this patch
            if self.aggregate_features:
                aggregated = torch.cat(cycle_features, dim=-1)  # [B, C * (cycles+1)]
                patch_feat = self.feature_aggregator(aggregated)  # [B, C]
            else:
                patch_feat = z_l

            patch_features.append(patch_feat)

        # Stack all patch features
        features = torch.stack(patch_features, dim=1)  # [B, N, C]

        # Final projection
        features = self.output_proj(features)

        return features

# ============================================================================
# Patchify Layer
# ============================================================================
class Patchify(nn.Module):
    def __init__(self, embed_dim, patch_size=4, in_channels=3):
        super().__init__()
        self.patch_size = patch_size
        self.conv1 = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, img):
        patches = self.conv1(img)  # [B, C, H', W']
        B, C, H, W = patches.shape
        patches = patches.permute(0, 2, 3, 1)  # [B, H', W', C]
        patches = self.norm(patches)
        return patches

# ============================================================================
# Multi-Block HRM Encoder
# ============================================================================
class MultiBlockHRMEncoder(nn.Module):
    """
    Enhanced encoder with multiple HRM blocks for hierarchical feature extraction
    """
    def __init__(self, num_blocks, h_cycle, l_cycle, embed_dim, patch_size, in_channels=3):
        super().__init__()
        self.num_blocks = num_blocks
        self.embed_dim = embed_dim

        # Initial patchification
        self.patcher = Patchify(embed_dim, patch_size=patch_size, in_channels=in_channels)

        # Stack of HRM blocks
        self.hrm_blocks = nn.ModuleList([
            HRMBlock(embed_dim, h_cycle, l_cycle, aggregate_features=True)
            for _ in range(num_blocks)
        ])

        # Inter-block projections with residual connections
        self.inter_block_projs = nn.ModuleList([
            nn.Sequential(
                nn.LayerNorm(embed_dim),
                nn.Linear(embed_dim, embed_dim),
                nn.GELU(),
                nn.Linear(embed_dim, embed_dim)
            )
            for _ in range(num_blocks - 1)
        ])

        # Final output projection
        self.final_norm = nn.LayerNorm(embed_dim)

    def forward(self, img):
        # Get patches
        patches = self.patcher(img)  # [B, H', W', C]
        B, H, W, C = patches.shape

        # Flatten spatial dimensions
        patches_flat = patches.reshape(B, H*W, C)  # [B, N, C]

        # Process through HRM blocks
        x = patches_flat
        for i, hrm_block in enumerate(self.hrm_blocks):
            # HRM processing
            x_hrm = hrm_block(x)

            # Residual connection
            x = x + x_hrm

            # Inter-block projection (except for last block)
            if i < self.num_blocks - 1:
                x = x + self.inter_block_projs[i](x)

        # Final normalization
        x = self.final_norm(x)

        # Reshape back to spatial format
        features = x.reshape(B, H, W, C)

        return features

# ============================================================================
# Enhanced Linear Decoder with Residual Blocks
# ============================================================================
class ResidualLinearBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.LayerNorm(channels),
            nn.Linear(channels, channels * 2),
            nn.GELU(),
            nn.Linear(channels * 2, channels)
        )

    def forward(self, x):
        return x + self.block(x)

class EnhancedLinearDecoder(nn.Module):
    """
    Enhanced decoder with:
    - Multiple residual blocks for feature processing
    - Progressive upsampling with residual connections
    - Better capacity through wider intermediate layers
    """
    def __init__(self, embed_dim, out_channels=3, img_size=32, latent_size=8, num_res_blocks=3):
        super().__init__()
        self.img_size = img_size
        self.latent_size = latent_size

        # Initial feature processing with residual blocks
        self.pre_process = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 2),
            nn.LayerNorm(embed_dim * 2),
            nn.GELU()
        )

        # Residual blocks for feature refinement
        self.res_blocks = nn.ModuleList([
            ResidualLinearBlock(embed_dim * 2)
            for _ in range(num_res_blocks)
        ])

        # Project to initial conv channels
        self.to_conv = nn.Linear(embed_dim * 2, 256)

        # Calculate upsampling stages
        scale_factor = img_size // latent_size
        num_upsamples = int(np.log2(scale_factor))

        # Build upsampling layers with residual connections
        upsample_layers = []
        in_ch = 256

        for i in range(num_upsamples):
            out_ch = max(64, in_ch // 2)

            upsample_layers.extend([
                nn.ConvTranspose2d(in_ch, out_ch, kernel_size=4, stride=2, padding=1),
                nn.GroupNorm(min(8, out_ch), out_ch),
                nn.GELU(),
                # Add residual conv block
                nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
                nn.GroupNorm(min(8, out_ch), out_ch),
                nn.GELU(),
            ])
            in_ch = out_ch

        # Final refinement layers
        upsample_layers.extend([
            nn.Conv2d(in_ch, in_ch, kernel_size=3, padding=1),
            nn.GroupNorm(min(8, in_ch), in_ch),
            nn.GELU(),
            nn.Conv2d(in_ch, out_channels, kernel_size=3, padding=1),
            nn.Tanh()
        ])

        self.upsample = nn.Sequential(*upsample_layers)

    def forward(self, z):
        # z: [B, H, W, C]
        B, H, W, C = z.shape

        # Process features with residual blocks
        z_flat = z.reshape(B * H * W, C)
        x = self.pre_process(z_flat)

        for res_block in self.res_blocks:
            x = res_block(x)

        x = self.to_conv(x)
        x = x.reshape(B, H, W, -1)

        # Convert to conv format [B, C, H, W]
        x = x.permute(0, 3, 1, 2)

        # Upsample to target size
        x = self.upsample(x)

        return x

# ============================================================================
# Complete Enhanced HRM VQ-VAE Model
# ============================================================================
class EnhancedHRM_VQVAE(nn.Module):
    def __init__(self, num_encoder_blocks=4, num_res_blocks=4,
                 h_cycle=3, l_cycle=2, embed_dim=256, patch_size=4,
                 num_embeddings=512, in_channels=1, out_channels=1):
        super().__init__()

        self.encoder = MultiBlockHRMEncoder(
            num_blocks=num_encoder_blocks,
            h_cycle=h_cycle,
            l_cycle=l_cycle,
            embed_dim=embed_dim,
            patch_size=patch_size,
            in_channels=in_channels
        )

        self.vq = VectorQuantizer(num_embeddings, embed_dim)

        img_size = 32  # For MNIST
        latent_size = img_size // patch_size

        self.decoder = EnhancedLinearDecoder(
            embed_dim=embed_dim,
            out_channels=out_channels,
            img_size=img_size,
            latent_size=latent_size,
            num_res_blocks=num_res_blocks
        )

    def forward(self, x):
        z_e = self.encoder(x)
        z_q, vq_loss, indices = self.vq(z_e)
        x_recon = self.decoder(z_q)
        return x_recon, vq_loss, indices

    def encode(self, x):
        z_e = self.encoder(x)
        _, _, indices = self.vq(z_e)
        return indices

    def decode_indices(self, indices):
        B, H, W = indices.shape
        quantized = self.vq.embeddings(indices.long())
        x_recon = self.decoder(quantized)
        return x_recon

# ============================================================================
# Training Script
# ============================================================================
def train_enhanced_vqvae(epochs=50, batch_size=128, lr=1e-3):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    # Dataset
    transform = transforms.Compose([
        transforms.Resize(32),
        transforms.ToTensor(),
        transforms.Normalize([0.5], [0.5])
    ])

    train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
    test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    # Initialize enhanced model
    model = EnhancedHRM_VQVAE(
        num_encoder_blocks=4,  # More HRM blocks
        num_res_blocks=4,      # More residual blocks in decoder
        h_cycle=3,
        l_cycle=2,
        embed_dim=256,
        patch_size=4,
        num_embeddings=512,
        in_channels=1,
        out_channels=1
    ).to(device)

    # Count parameters
    total_params = sum(p.numel() for p in model.parameters())
    encoder_params = sum(p.numel() for p in model.encoder.parameters())
    decoder_params = sum(p.numel() for p in model.decoder.parameters())

    print(f"\n{'='*60}")
    print(f"Enhanced Multi-Block HRM VQ-VAE")
    print(f"{'='*60}")
    print(f"Total parameters: {total_params:,}")
    print(f"Encoder parameters: {encoder_params:,}")
    print(f"Decoder parameters: {decoder_params:,}")
    print(f"Encoder blocks: 4 | Decoder residual blocks: 4")
    print(f"{'='*60}\n")

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, epochs)

    best_loss = float('inf')

    # Training loop
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        train_recon_loss = 0
        train_vq_loss = 0

        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}')
        for batch_idx, (data, _) in enumerate(pbar):
            data = data.to(device)

            optimizer.zero_grad()
            recon, vq_loss, _ = model(data)

            recon_loss = F.mse_loss(recon, data)
            loss = recon_loss + vq_loss

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            train_loss += loss.item()
            train_recon_loss += recon_loss.item()
            train_vq_loss += vq_loss.item()

            pbar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'recon': f'{recon_loss.item():.4f}',
                'vq': f'{vq_loss.item():.4f}'
            })

        scheduler.step()

        avg_loss = train_loss / len(train_loader)
        avg_recon = train_recon_loss / len(train_loader)
        avg_vq = train_vq_loss / len(train_loader)

        print(f'\nEpoch {epoch+1}: Loss={avg_loss:.4f}, Recon={avg_recon:.4f}, VQ={avg_vq:.4f}')

        # Save best model
        if avg_loss < best_loss:
            best_loss = avg_loss
            torch.save(model.state_dict(), 'enhanced_hrm_vqvae_best.pth')
            print(f"✓ Saved new best model (loss: {best_loss:.4f})")

        # Visualize every 5 epochs
        if (epoch + 1) % 5 == 0:
            visualize_reconstructions(model, test_loader, device, epoch+1)

    return model

def visualize_reconstructions(model, test_loader, device, epoch):
    model.eval()
    with torch.no_grad():
        data, _ = next(iter(test_loader))
        data = data[:8].to(device)
        recon, _, indices = model(data)

        data = data.cpu() * 0.5 + 0.5
        recon = recon.cpu() * 0.5 + 0.5

        fig, axes = plt.subplots(2, 8, figsize=(16, 4))
        for i in range(8):
            axes[0, i].imshow(data[i].squeeze(), cmap='gray')
            axes[1, i].imshow(recon[i].squeeze(), cmap='gray')
            axes[0, i].axis('off')
            axes[1, i].axis('off')

        axes[0, 0].set_ylabel('Original', fontsize=12)
        axes[1, 0].set_ylabel('Reconstructed', fontsize=12)
        plt.tight_layout()
        plt.savefig(f'enhanced_recon_epoch_{epoch}.png', dpi=150, bbox_inches='tight')
        plt.close()
        print(f"Saved enhanced_recon_epoch_{epoch}.png")

        unique_codes = torch.unique(indices).numel()
        total_codes = model.vq.num_embeddings
        print(f"Codebook usage: {unique_codes}/{total_codes} ({100*unique_codes/total_codes:.1f}%)")

# ============================================================================
# Main Execution
# ============================================================================
if __name__ == '__main__':
    model = train_enhanced_vqvae(
        epochs=30,
        batch_size=64,
        lr=3e-4  # Start with higher LR, scheduler will reduce it
    )

    print("\n✓ Training complete!")
    print("Model saved as 'enhanced_hrm_vqvae_best.pth'")

Using device: cuda

Enhanced Multi-Block HRM VQ-VAE
Total parameters: 13,749,121
Encoder parameters: 8,271,104
Decoder parameters: 5,346,945
Encoder blocks: 4 | Decoder residual blocks: 4



Epoch 1/30: 100%|██████████| 938/938 [12:14<00:00,  1.28it/s, loss=0.4626, recon=0.2519, vq=0.2107]



Epoch 1: Loss=0.9489, Recon=0.1404, VQ=0.8084
✓ Saved new best model (loss: 0.9489)


Epoch 2/30: 100%|██████████| 938/938 [12:20<00:00,  1.27it/s, loss=0.2692, recon=0.2676, vq=0.0016]



Epoch 2: Loss=0.3160, Recon=0.2420, VQ=0.0739
✓ Saved new best model (loss: 0.3160)


...

Epoch 11/30: 100%|██████████| 938/938 [12:13<00:00,  1.28it/s, loss=0.2390, recon=0.2390, vq=0.0001]



Epoch 11: Loss=0.2419, Recon=0.2418, VQ=0.0001
✓ Saved new best model (loss: 0.2419)


Epoch 12/30: 100%|██████████| 938/938 [12:10<00:00,  1.28it/s, loss=0.2450, recon=0.2449, vq=0.0001]



Epoch 12: Loss=0.2419, Recon=0.2418, VQ=0.0001


Epoch 13/30:  22%|██▏       | 204/938 [02:39<10:23,  1.18it/s, loss=0.2244, recon=0.2243, vq=0.0001]

In [ ]:
model.named_modules